<a href="https://colab.research.google.com/github/jstyoon96/WPI-AI-Course/blob/main/WPI_week1/lab2/WPI_week1_lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Biomedical Signals: ECG Rule-Based Baseline

**Audience:** WPI AI Bootcamp students with basic Python experience.

**Estimated time:** 90-120 minutes.

**Clinical disclaimer:** The event, risk score, and alarm in this lab are synthetic teaching outputs for workflow practice. They are not clinically validated measurements and must not be used for diagnosis.


## Learning Objectives

By the end of this lab, you should be able to:

- Treat an ECG as a time-indexed biomedical signal.
- Apply simple preprocessing before rule-based analysis.
- Detect candidate R-peaks and estimate instantaneous heart rate.
- Build a transparent rule-based screening baseline.
- Convert a signal workflow into structured output fields.


## Grading And Word Response Submission

This lab is graded out of **100 pts**.

- Notebook execution and artifacts: **60 pts**
- Word response document: **40 pts**

Use this filename for the Word response document:

`WPI_week1_lab2_responses_LastName_FirstName.docx`

Write Q1-Q5 in the Word document, using 2-5 sentences per question. Keep longer written responses in the Word document rather than in notebook markdown cells. The notebook should contain the generated plots, tables, and short notes requested in each part.


## Workflow

This lab follows a classic signal baseline pipeline:

`ECG -> Inspect -> Preprocess -> Detect Peaks -> Estimate HR -> Rule Screen -> QC -> Structured Output`

The rule-based baseline is intentionally simple and interpretable. It is not an AI model and it is not a clinical detector.


## Setup

Run this setup cell first. The notebook installs the small set of packages used in the lab, clones the public course helper repo in Colab, and imports shared data/style helpers.


In [ ]:
#@title Setup course environment
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("XDG_CACHE_HOME", "/tmp/.cache")

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "scipy",
    "pooch",
    "numpy",
    "pandas",
    "matplotlib",
])

repo_dir = Path("/content/WPI-AI-Course")
if not repo_dir.parent.exists():
    local_repo = Path.cwd()
    repo_dir = local_repo if (local_repo / "wpi_ai_bootcamp").exists() else Path("/tmp/WPI-AI-Course")

if not repo_dir.exists():
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/jstyoon96/WPI-AI-Course.git",
        str(repo_dir),
    ])
elif (repo_dir / ".git").exists() and repo_dir.name == "WPI-AI-Course":
    subprocess.check_call(["git", "-C", str(repo_dir), "pull", "--ff-only"])

sys.path.insert(0, str(repo_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

from wpi_ai_bootcamp.data import load_ecg_signal
from wpi_ai_bootcamp.notebook import setup_lab
from wpi_ai_bootcamp.signals import detect_r_peaks_pan_tompkins

WPI_COLORS = setup_lab()
print("Setup complete.")




## Data Loading

This lab does not use a committed ECG data file. The raw ECG is loaded at runtime from SciPy sample data through `wpi_ai_bootcamp.data.load_ecg_signal()`.

To make the rule-based baseline visible in a short lab window, we create a clearly labeled synthetic event segment inside a teaching copy of the ECG. The synthetic segment is for education only and is not clinical ground truth.


In [ ]:
ecg, fs, source = load_ecg_signal()
ecg = np.asarray(ecg, dtype=np.float32)

T_TOTAL = 30
n = int(T_TOTAL * fs)
ecg = ecg[:n]
t = np.arange(len(ecg)) / fs

print(source.name)
print(source.url)
print("sampling rate fs:", fs, "Hz")
print("duration:", len(ecg) / fs, "seconds")
print("shape:", ecg.shape, "dtype:", ecg.dtype)

source_table = pd.DataFrame([
    {
        "name": source.name,
        "loader": source.loader,
        "citation": source.citation,
        "license_note": source.license_note,
    }
])
source_table


## Part 1 — Inspect The Raw ECG

ECG samples are measured over time. Plotting against seconds makes physiological timing easier to interpret than plotting against sample index.


In [ ]:
def normalize_unit(signal_values):
    values = np.asarray(signal_values, dtype=np.float32)
    scale = np.percentile(np.abs(values), 95)
    return values / (scale + 1e-8)

ecg_raw = normalize_unit(ecg)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t, ecg_raw, color=WPI_COLORS["crimson"], linewidth=1.0)
axes[0].set_title("Raw ECG teaching window")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Normalized amplitude")

zoom_mask = (t >= 0) & (t <= 8)
axes[1].plot(t[zoom_mask], ecg_raw[zoom_mask], color=WPI_COLORS["dark_gray"], linewidth=1.0)
axes[1].set_title("First 8 seconds")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Normalized amplitude")
plt.show()

print("raw ECG range:", float(ecg_raw.min()), float(ecg_raw.max()))


### Part 1 Assessment — Raw ECG Inspection (20 pts)

- Notebook artifact: show the full 30-second ECG window and a zoomed 0-8 second view. **12 pts**
- Word response Q1: Why is ECG plotted against time instead of sample index? **8 pts**
- Grading criteria: the notebook should clearly label time in seconds; the Word response should connect sampling rate to physiological interpretation.
- TODO: After running the plots, write one short notebook note naming one visible ECG feature such as QRS spikes, baseline drift, or noise.


## Part 2 — Preprocessing

A simple bandpass filter can reduce slow baseline drift and high-frequency noise. Filtering is helpful, but over-filtering can distort morphology that later rules depend on.


In [ ]:
LOW_CUT_HZ = 0.5
HIGH_CUT_HZ = 40.0

sos = signal.butter(3, [LOW_CUT_HZ, HIGH_CUT_HZ], btype="bandpass", fs=fs, output="sos")
ecg_filtered_raw = signal.sosfiltfilt(sos, ecg_raw)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t[zoom_mask], ecg_raw[zoom_mask], label="Raw", color=WPI_COLORS["gray"], linewidth=1.0)
axes[0].plot(t[zoom_mask], ecg_filtered_raw[zoom_mask], label="Filtered", color=WPI_COLORS["crimson"], linewidth=1.0)
axes[0].set_title("Raw vs filtered ECG")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")
axes[0].legend()

residual = ecg_raw - ecg_filtered_raw
axes[1].plot(t[zoom_mask], residual[zoom_mask], color=WPI_COLORS["dark_gray"], linewidth=1.0)
axes[1].set_title("Removed component")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Raw - filtered")
plt.show()

print("filter band:", LOW_CUT_HZ, "to", HIGH_CUT_HZ, "Hz")


### Part 2 Assessment — Preprocessing (20 pts)

- Notebook artifact: show raw vs filtered ECG and the removed component. **12 pts**
- Word response Q2: What changed after preprocessing, and what risk comes from over-filtering? **8 pts**
- Grading criteria: the notebook should compare signals on the same time window; the Word response should mention noise/drift reduction and possible morphology distortion.
- TODO: Try changing `LOW_CUT_HZ` or `HIGH_CUT_HZ` once, then rerun this section and compare the filtered signal.


## Teaching Event Injection

The public SciPy ECG sample is useful for signal processing, but a short lab needs an event that is easy to inspect. Here we create a teaching copy of the ECG with one synthetic tachy/wide-complex-like segment. This synthetic segment is only for learning how a rule-based workflow behaves.


In [ ]:
EVENT_START_S = 12.0
EVENT_DURATION_S = 4.0
EVENT_END_S = EVENT_START_S + EVENT_DURATION_S

ecg_teaching = ecg_raw.copy()
event_mask = (t >= EVENT_START_S) & (t < EVENT_END_S)
event_t = t[event_mask] - EVENT_START_S

synthetic_event = np.zeros_like(event_t)
synthetic_beat_times = np.arange(0.15, EVENT_DURATION_S, 0.42)
for beat_time in synthetic_beat_times:
    synthetic_event += 1.6 * np.exp(-0.5 * ((event_t - beat_time) / 0.035) ** 2)
    synthetic_event -= 0.35 * np.exp(-0.5 * ((event_t - (beat_time + 0.08)) / 0.040) ** 2)
synthetic_event += 0.05 * np.sin(2 * np.pi * 18 * event_t)
ecg_teaching[event_mask] = synthetic_event.astype(np.float32)

ecg_filtered = signal.sosfiltfilt(sos, ecg_teaching)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(t, ecg_teaching, color=WPI_COLORS["dark_gray"], linewidth=0.9)
ax.axvspan(EVENT_START_S, EVENT_END_S, color=WPI_COLORS["crimson"], alpha=0.20, label="Synthetic teaching event")
ax.set_title("Teaching ECG with synthetic event")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Normalized amplitude")
ax.legend()
plt.show()

print("synthetic event window:", EVENT_START_S, "to", EVENT_END_S, "seconds")
print("synthetic beats inserted:", len(synthetic_beat_times))


## Part 3 — Pan-Tompkins Style R-Peak Detection And Heart Rate

A Pan-Tompkins style detector emphasizes QRS complexes before choosing candidate peaks. The teaching implementation used here applies QRS bandpass filtering, derivative, squaring, moving-window integration, adaptive thresholding, and local R-peak refinement. It is transparent and useful for learning, but it is not a certified clinical detector.


In [ ]:
THRESHOLD_SCALE = 0.20
REFRACTORY_S = 0.200

peak_result = detect_r_peaks_pan_tompkins(
    ecg_teaching,
    fs,
    threshold_scale=THRESHOLD_SCALE,
    refractory_s=REFRACTORY_S,
)

r_locs = peak_result["r_locs"]
r_times = peak_result["r_times"]
hr = peak_result["hr"]
hr_t = peak_result["hr_t"]
qrs_bandpassed = peak_result["bandpassed"]
qrs_integrated = peak_result["integrated"]
r_threshold = peak_result["threshold"]

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(t, ecg_filtered, color=WPI_COLORS["dark_gray"], linewidth=0.9)
axes[0].scatter(r_times, ecg_filtered[r_locs], color=WPI_COLORS["crimson"], s=18, label="Detected R-peaks")
axes[0].axvspan(EVENT_START_S, EVENT_END_S, color=WPI_COLORS["crimson"], alpha=0.12)
axes[0].set_title("Filtered ECG with Pan-Tompkins style R-peaks")
axes[0].set_ylabel("Amplitude")
axes[0].legend()

axes[1].plot(t, qrs_integrated, color=WPI_COLORS["dark_gray"], linewidth=0.9, label="Moving-window integration")
axes[1].axhline(r_threshold, color=WPI_COLORS["crimson"], linestyle="--", label="Adaptive threshold")
axes[1].axvspan(EVENT_START_S, EVENT_END_S, color=WPI_COLORS["crimson"], alpha=0.12)
axes[1].set_title("Pan-Tompkins intermediate signal")
axes[1].set_ylabel("Integrated energy")
axes[1].legend()

axes[2].plot(hr_t, hr, color=WPI_COLORS["crimson"], marker="o", markersize=3, linewidth=1.0)
axes[2].axhline(120, color=WPI_COLORS["dark_gray"], linestyle="--", label="120 bpm reference")
axes[2].axvspan(EVENT_START_S, EVENT_END_S, color=WPI_COLORS["crimson"], alpha=0.12)
axes[2].set_title("Instantaneous heart rate")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("HR (bpm)")
axes[2].legend()
plt.show()

print("detected peaks:", len(r_locs))
print("adaptive threshold:", r_threshold)
print("threshold scale:", THRESHOLD_SCALE)
print("refractory window:", REFRACTORY_S, "seconds")
print("HR range:", float(np.nanmin(hr)), "to", float(np.nanmax(hr)), "bpm")


### Part 3 Assessment — Pan-Tompkins Style R-Peak Detection And HR (20 pts)

- Notebook artifact: show detected peaks, moving-window integration with adaptive threshold, and the instantaneous HR curve. **12 pts**
- Word response Q3: How does R-peak detection connect to heart-rate estimation? **8 pts**
- Grading criteria: the notebook should show peak markers and a plausible HR curve; the Word response should connect R-R interval duration to beats per minute.
- TODO: Change `THRESHOLD_SCALE` or `REFRACTORY_S` once and note whether the detector finds more or fewer peaks.


## Part 4 — Rule-Based Event Screening

A transparent baseline can combine simple measurements. Here we flag candidate event intervals when heart rate is high and local signal energy is high. This is a teaching rule, not a clinical VT detector.


In [ ]:
HR_THRESHOLD = 120.0
ENERGY_PERCENTILE = 85
MIN_CONSECUTIVE_FLAGS = 2

energy_window = max(1, int(0.250 * fs))
energy = np.sqrt(np.convolve(ecg_filtered ** 2, np.ones(energy_window) / energy_window, mode="same"))
energy_at_beats = energy[r_locs[1:]]
energy_threshold = float(np.percentile(energy, ENERGY_PERCENTILE))

candidate_flags = (hr > HR_THRESHOLD) & (energy_at_beats > energy_threshold)

segments = []
flag_idx = np.where(candidate_flags)[0]
if len(flag_idx) > 0:
    groups = np.split(flag_idx, np.where(np.diff(flag_idx) > 1)[0] + 1)
    for group in groups:
        start_s = float(hr_t[group[0]])
        end_s = float(hr_t[group[-1]])
        if len(group) >= MIN_CONSECUTIVE_FLAGS or (end_s - start_s) >= 1.0:
            segments.append((start_s, end_s))

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
axes[0].plot(t, ecg_filtered, color=WPI_COLORS["dark_gray"], linewidth=0.9)
axes[0].axvspan(EVENT_START_S, EVENT_END_S, color=WPI_COLORS["crimson"], alpha=0.12, label="Synthetic event")
for start_s, end_s in segments:
    axes[0].axvspan(start_s, end_s, color=WPI_COLORS["crimson"], alpha=0.28)
axes[0].set_title("Rule-screened ECG segments")
axes[0].set_ylabel("Filtered ECG")
axes[0].legend()

axes[1].plot(hr_t, hr, color=WPI_COLORS["crimson"], marker="o", markersize=3, linewidth=1.0, label="HR")
axes[1].axhline(HR_THRESHOLD, color=WPI_COLORS["dark_gray"], linestyle="--", label="HR threshold")
axes[1].scatter(hr_t[candidate_flags], hr[candidate_flags], color="black", s=28, label="Rule flags")
axes[1].set_title("Candidate flags from HR + energy rule")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("HR (bpm)")
axes[1].legend()
plt.show()

print("energy threshold:", energy_threshold)
print("candidate flags:", int(candidate_flags.sum()))
print("segments:", segments)


### Part 4 Assessment — Rule-Based Screening (20 pts)

- Notebook artifact: show flagged points/segments and print the rule thresholds. **12 pts**
- Word response Q4: What rule did the baseline use to flag the synthetic event, and where could it fail? **8 pts**
- Grading criteria: the notebook should show the synthetic event region and candidate flags; the Word response should mention at least one failure mode such as noise, missed peaks, or threshold sensitivity.
- TODO: Try one threshold change, such as `HR_THRESHOLD` or `ENERGY_PERCENTILE`, and observe whether the flags change.


## Part 5 — Visual QC And Structured Output

A useful workflow produces both a visual check and a structured table. The table below uses the same public schema as the imaging lab: `risk_score`, `alarm`, `quality`, and `review_flag`.


In [ ]:
alarm = int(len(segments) > 0)
quality_proxy = np.std(ecg_filtered) / (np.std(ecg_teaching - ecg_filtered) + 1e-8)
quality = float(np.clip((quality_proxy - 1.0) / 9.0, 0, 1))

max_hr_score = float(np.clip((np.nanmax(hr) - 100.0) / 80.0, 0, 1))
energy_score = float(np.clip((np.percentile(energy_at_beats, 95) / (energy_threshold + 1e-8)) - 1.0, 0, 1))
risk_score = float(np.clip(0.6 * max_hr_score + 0.4 * energy_score, 0, 1))
review_flag = int(alarm == 1 or quality < 0.25)

output_table = pd.DataFrame([
    {
        "risk_score": risk_score,
        "alarm": alarm,
        "quality": quality,
        "review_flag": review_flag,
    }
])

segment_table = pd.DataFrame(segments, columns=["start_s", "end_s"])
if segment_table.empty:
    segment_table = pd.DataFrame([{"start_s": np.nan, "end_s": np.nan}])

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(t, ecg_filtered, color=WPI_COLORS["dark_gray"], linewidth=0.9)
ax.axvspan(EVENT_START_S, EVENT_END_S, color=WPI_COLORS["gray"], alpha=0.20, label="Synthetic event window")
for start_s, end_s in segments:
    ax.axvspan(start_s, end_s, color=WPI_COLORS["crimson"], alpha=0.30, label="Rule alarm segment")
ax.set_title("Final QC overlay")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Filtered ECG")
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys())
plt.show()

print("segment table")
print(segment_table.to_string(index=False))
print("structured output")
output_table


### Part 5 Assessment — Final QC Decision (20 pts)

- Notebook artifact: show the final QC overlay, `segment_table`, and `output_table`. **12 pts**
- Word response Q5: Based on plots and the output table, would you trust the alarm or request human review? **8 pts**
- Grading criteria: the notebook should include all required schema fields; the Word response should use both visual evidence and table values such as `risk_score`, `alarm`, `quality`, or `review_flag`.
- TODO: Use the final overlay and table values to choose whether the case needs human review.


## Word Response Prompts

Write responses to Q1-Q5 in `WPI_week1_lab2_responses_LastName_FirstName.docx`. Use 2-5 sentences per question.

1. Why is ECG plotted against time instead of sample index?
2. What changed after preprocessing, and what risk comes from over-filtering?
3. How does R-peak detection connect to heart-rate estimation?
4. What rule did the baseline use to flag the synthetic event, and where could it fail?
5. Based on plots and the output table, would you trust the alarm or request human review?


## Optional Challenge

Change `HR_THRESHOLD`, `ENERGY_PERCENTILE`, or `MIN_CONSECUTIVE_FLAGS` and rerun Part 4 and Part 5. How does the alarm behavior change? Which setting would you choose if false alarms are costly?


## Attribution

- Data: SciPy electrocardiogram sample, loaded through `scipy.datasets.electrocardiogram()` via `wpi_ai_bootcamp.data.load_ecg_signal()`.
- Source note: SciPy documents the sample as derived from MIT-BIH Arrhythmia Database record 208 on PhysioNet.
- Libraries: NumPy, pandas, Matplotlib, and SciPy.
- WPI colors: WPI Institutional Guidelines, Crimson `#AC2B37` and Gray `#A9B0B7`.
